In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources (if not already present)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

print("Imports successful!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hasan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hasan\AppData\Roaming\nltk_data...


Imports successful!


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hasan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Load the dataset (same as in EDA)
df = pd.read_csv('../data/raw/IMDB Dataset.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
def clean_text(text, remove_stopwords=True, use_stemming=True, use_lemmatization=False):
    """
    Clean and preprocess text data.
    
    Parameters:
    - text: raw text string
    - remove_stopwords: whether to remove stopwords
    - use_stemming: apply Porter stemming (if True)
    - use_lemmatization: apply WordNet lemmatization (if True, overrides stemming)
    
    Returns:
    - cleaned text string
    """
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 4. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 5. Tokenize
    words = text.split()
    
    # 6. Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        words = [w for w in words if w not in stop_words]
    
    # 7. Stemming or Lemmatization
    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(w) for w in words]
    elif use_stemming:
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]
    
    # 8. Join back into a single string
    return ' '.join(words)

In [4]:
sample_review = df['review'].iloc[0]
print("Original review:\n", sample_review)
print("\nCleaned review (no stopwords, stemming):\n", clean_text(sample_review, remove_stopwords=True, use_stemming=True))

Original review:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

In [5]:
# Apply cleaning (choose options)
print("Cleaning reviews... This may take a while.")
df['cleaned_review'] = df['review'].apply(lambda x: clean_text(x, remove_stopwords=True, use_stemming=True))
print("Done!")

# Show a few examples
df[['review', 'cleaned_review']].head()

Cleaning reviews... This may take a while.
Done!


,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product film techniqu unassum old...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [12]:
# Convert sentiment to binary: positive -> 1, negative -> 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(df[['sentiment', 'label']].head())

  sentiment  label
0  positive      1
1  positive      1
2  positive      1
3  negative      0
4  positive      1


In [13]:
# Some reviews might become empty after cleaning (e.g., if only stopwords)
empty_reviews = df[df['cleaned_review'] == '']
print(f"Number of empty reviews after cleaning: {len(empty_reviews)}")

# If any, we could drop them
if len(empty_reviews) > 0:
    df = df[df['cleaned_review'] != ''].reset_index(drop=True)
    print(f"Dataset shape after removing empty reviews: {df.shape}")

Number of empty reviews after cleaning: 0


In [14]:
# Separate features and labels
X = df['cleaned_review']
y = df['label']

# Split (80% train, 20% test) with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training set class distribution:\n{y_train.value_counts()}")
print(f"Test set class distribution:\n{y_test.value_counts()}")

Training set size: 40000
Test set size: 10000
Training set class distribution:
label
1    20000
0    20000
Name: count, dtype: int64
Test set class distribution:
label
0    5000
1    5000
Name: count, dtype: int64


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a small sample for demonstration
sample_texts = X_train.iloc[:5].tolist()
print("Sample texts:")
for i, txt in enumerate(sample_texts):
    print(f"{i+1}: {txt[:100]}...")  # show first 100 chars

# Initialize TF-IDF vectorizer with some parameters
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

# Fit and transform the sample
sample_tfidf = tfidf.fit_transform(sample_texts)

# Show feature names (top 1000 words)
print("\nFeature names (first 20):", tfidf.get_feature_names_out()[:20])

# Show TF-IDF matrix shape
print(f"\nTF-IDF matrix shape: {sample_tfidf.shape}")

Sample texts:
1: caught littl gem total accid back reviv theatr see two old silli scifi movi theatr pack full warn sh...
2: cant believ let movi accomplish favor friend ask earli april movi certainli pain ass theater sickli ...
3: spoiler alert get nerv peopl remak use term loos good movi american version dutch thriller someon de...
4: there one thing ive learnt watch georg romero creepshow stumbl upon mysteri old crate someon obvious...
5: rememb theater review said horribl well didnt think bad amus lot tongueincheek humor concern famili ...

Feature names (first 20): ['absolut' 'accid' 'accomplish' 'act' 'advis' 'affleck' 'alert' 'allalso'
 'alon' 'american' 'amus' 'ancient' 'anger' 'anim' 'anymoreth' 'anyon'
 'anyth' 'appropri' 'april' 'arent']

TF-IDF matrix shape: (5, 354)


In [16]:
# Save full cleaned dataset
df.to_csv('../data/processed/cleaned_reviews.csv', index=False)

# Save train/test splits as CSV
train_df = pd.DataFrame({'review': X_train, 'label': y_train})
test_df = pd.DataFrame({'review': X_test, 'label': y_test})

train_df.to_csv('../data/processed/train.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print("Processed data saved to ../data/processed/")

Processed data saved to ../data/processed/


In [11]:
print("Preprocessing complete!")
print("- Cleaned reviews with stopword removal and stemming")
print("- Labels encoded (positive=1, negative=0)")
print("- Data split into train/test (80/20)")
print("- Ready for modeling in the next notebook")

Preprocessing complete!
- Cleaned reviews with stopword removal and stemming
- Labels encoded (positive=1, negative=0)
- Data split into train/test (80/20)
- Ready for modeling in the next notebook
